# Task 3: Airborne and Satellite Data Integration for Water Quality
This notebook focuses on the comparison between high-resolution airborne hyperspectral data and Sentinel-2 multispectral data. We will:
1. Load and visualize airborne data.
2. Calculate water quality indices (Chl-a, DOC, Turbidity).
3. Fetch corresponding Sentinel-2 data for the same date (**2025-06-17**).
4. Compare the results between both sensors visually and statistically.

### Step 1: Environment Setup
We import necessary libraries for data processing (numpy, spectral), spatial analysis (rasterio, pystac), and visualization (matplotlib).

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import spectral.io.envi as envi
from pystac_client import Client
import planetary_computer
from datetime import datetime

# Set acquisition date
ACQUISITION_DATE = '2025-06-17'

# Relative path to airborne data
BASE_DIR = Path('data/you-shall-not-pass/Obrazy lotnicze')
hdr_path = BASE_DIR / '221000_Odra_HS_Blok_A_008_VS_join_atm.hdr'

print(f"Target Date: {ACQUISITION_DATE}")
print(f"Airborne Header Path: {hdr_path}")

### Step 2: Load Airborne Data Cube
We open the ENVI hyperspectral file and load a subset of the data into memory. We also extract the wavelength information associated with each band.

In [ ]:
if not hdr_path.exists():
    raise FileNotFoundError(f"Cannot find airborne data at {hdr_path}")

img = envi.open(hdr_path)
meta = img.metadata
wavelengths = np.array([float(x) for x in meta['wavelength']])

# Read a subset (1000x1000) from the center to keep memory usage low
cy, cx = img.nrows // 2, img.ncols // 2
cube = img.read_subregion((cy-500, cy+500), (cx-500, cx+500))
print(f"Loaded subset shape: {cube.shape}")

### Step 3: Display False-Color Composite
We select bands corresponding to Red (~670nm), Green (~560nm), and Blue (~490nm) to create a standard RGB visualization of the airborne cube.

In [ ]:
r_idx = np.argmin(np.abs(wavelengths - 670))
g_idx = np.argmin(np.abs(wavelengths - 560))
b_idx = np.argmin(np.abs(wavelengths - 490))

rgb_air = cube[:, :, [r_idx, g_idx, b_idx]]
# Simple stretch for visualization
rgb_air = (rgb_air - np.percentile(rgb_air, 2)) / (np.percentile(rgb_air, 98) - np.percentile(rgb_air, 2))
rgb_air = np.clip(rgb_air, 0, 1)

plt.figure(figsize=(8, 8))
plt.imshow(rgb_air)
plt.title("Airborne False Color RGB")
plt.axis('off')
plt.show()

### Step 4: Calculate Water Quality Indices (Airborne)
We compute ratios typically used for water quality:
- **Chl-a**: Ratio of 705nm / 665nm (Red edge / Red)
- **DOC**: Ratio of 560nm / 665nm (Green / Red)
- **Turbidity**: Reflected intensity at 700nm.

In [ ]:
def safe_ratio(n, d, eps=1e-6):
    return n / (d + eps)

idx_705 = np.argmin(np.abs(wavelengths - 705))
idx_665 = np.argmin(np.abs(wavelengths - 665))
idx_560 = np.argmin(np.abs(wavelengths - 560))
idx_700 = np.argmin(np.abs(wavelengths - 700))

chla_air = safe_ratio(cube[:, :, idx_705], cube[:, :, idx_665])
doc_air = safe_ratio(cube[:, :, idx_560], cube[:, :, idx_665])
turb_air = cube[:, :, idx_700]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, (data, name) in enumerate(zip([chla_air, doc_air, turb_air], ['Chl-a', 'DOC', 'Turbidity'])):
    im = axes[i].imshow(data, cmap='viridis')
    axes[i].set_title(f"Airborne {name}")
    plt.colorbar(im, ax=axes[i])
plt.show()

### Step 5: Download Sentinel-2 Data
We connect to the Microsoft Planetary Computer STAC API to find a Sentinel-2 scene covering the Odra river region on **2025-06-17**.

In [ ]:
catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1", modifier=planetary_computer.sign_inplace)

# ROI: Odra river coordinates near the airborne flight
bbox = [14.4, 52.8, 14.8, 53.2]

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=ACQUISITION_DATE,
    query={"eo:cloud_cover": {"lt": 20}}
)

items = list(search.items())
if not items:
    print("No Sentinel-2 scenes found for the specific date. Checking nearest...")
    search = catalog.search(collections=["sentinel-2-l2a"], bbox=bbox, datetime="2025-06-10/2025-06-25", sortby="datetime")
    items = list(search.items())

best_item = items[0]
print(f"Selected Sentinel-2 Scene: {best_item.id} ({best_item.datetime})")

# We prepare the URLs for the required bands
assets = best_item.assets
bands_s2 = {
    'B03': assets['B03'].href, # Green
    'B04': assets['B04'].href, # Red
    'B05': assets['B05'].href, # Red Edge 1 (~705nm)
    'B11': assets['B11'].href  # SWIR 1 (used for turbidity proxy)
}

### Step 6: Calculate Water Quality Indices (Sentinel-2)
We stream the Sentinel-2 bands and calculate the same indices as for the airborne data. We use B05 for the 705nm equivalent and B03/B04 for Green/Red. SWIR (B11) is used as a proxy for turbidity.

In [ ]:
data_s2 = {}
with rasterio.open(bands_s2['B04']) as src:
    # Read a 500x500 window corresponding to the general area
    win = rasterio.windows.Window(1000, 1000, 500, 500)
    data_s2['B04'] = src.read(1, window=win).astype(float) / 10000.0

with rasterio.open(bands_s2['B03']) as src:
    data_s2['B03'] = src.read(1, window=win).astype(float) / 10000.0

with rasterio.open(bands_s2['B05']) as src:
    # B05 is 20m resolution, upsample to match 10m grid
    data_s2['B05'] = src.read(1, out_shape=data_s2['B04'].shape, resampling=rasterio.enums.Resampling.bilinear).astype(float) / 10000.0

with rasterio.open(bands_s2['B11']) as src:
    # B11 is 20m resolution, upsample to match 10m grid
    data_s2['B11'] = src.read(1, out_shape=data_s2['B04'].shape, resampling=rasterio.enums.Resampling.bilinear).astype(float) / 10000.0

chla_s2 = safe_ratio(data_s2['B05'], data_s2['B04'])
doc_s2 = safe_ratio(data_s2['B03'], data_s2['B04'])
turb_s2 = data_s2['B11']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(chla_s2, cmap='viridis', vmin=0, vmax=2)
axes[0].set_title("Sentinel-2 Chl-a Proxy")
axes[1].imshow(doc_s2, cmap='viridis', vmin=0, vmax=2)
axes[1].set_title("Sentinel-2 DOC Proxy")
axes[2].imshow(turb_s2, cmap='viridis')
axes[2].set_title("Sentinel-2 Turbidity Proxy")
plt.show()

### Step 7: Compare Results (Values)
We compare the mean values of the indices from the airborne subset and the Sentinel-2 patch. Note that differences are expected due to resolution (1m vs 10m) and atmospheric correction variations.

In [ ]:
print("--- Comparison of Mean Index Values ---")
print(f"Airborne Chl-a: {np.nanmean(chla_air):.4f}")
print(f"Sentinel-2 Chl-a: {np.nanmean(chla_s2):.4f}")
print(f"Relative Difference (Chl-a): {abs(np.nanmean(chla_air)-np.nanmean(chla_s2))/np.nanmean(chla_air)*100:.2f}%")

print(f"\nAirborne DOC: {np.nanmean(doc_air):.4f}")
print(f"Sentinel-2 DOC: {np.nanmean(doc_s2):.4f}")
print(f"Relative Difference (DOC): {abs(np.nanmean(doc_air)-np.nanmean(doc_s2))/np.nanmean(doc_air)*100:.2f}%")

print(f"\nAirborne Turbidity: {np.nanmean(turb_air):.4f}")
print(f"Sentinel-2 Turbidity: {np.nanmean(turb_s2):.4f}")
print(f"Relative Difference (Turbidity): {abs(np.nanmean(turb_air)-np.nanmean(turb_s2))/np.nanmean(turb_air)*100:.2f}%")

### Step 8: Visual Comparison Side-by-Side
We plot the indices from both sensors side-by-side to visually assess the spatial distribution and patterns. Note the resolution and footprint differences.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 18))

# Chl-a
im0 = axes[0, 0].imshow(chla_air, cmap='viridis', vmin=0, vmax=2)
axes[0, 0].set_title("Airborne Chl-a")
plt.colorbar(im0, ax=axes[0, 0])
im1 = axes[0, 1].imshow(chla_s2, cmap='viridis', vmin=0, vmax=2)
axes[0, 1].set_title("Sentinel-2 Chl-a")
plt.colorbar(im1, ax=axes[0, 1])

# DOC
im2 = axes[1, 0].imshow(doc_air, cmap='viridis', vmin=0, vmax=2)
axes[1, 0].set_title("Airborne DOC")
plt.colorbar(im2, ax=axes[1, 0])
im3 = axes[1, 1].imshow(doc_s2, cmap='viridis', vmin=0, vmax=2)
axes[1, 1].set_title("Sentinel-2 DOC")
plt.colorbar(im3, ax=axes[1, 1])

# Turbidity
im4 = axes[2, 0].imshow(turb_air, cmap='viridis')
axes[2, 0].set_title("Airborne Turbidity")
plt.colorbar(im4, ax=axes[2, 0])
im5 = axes[2, 1].imshow(turb_s2, cmap='viridis')
axes[2, 1].set_title("Sentinel-2 Turbidity")
plt.colorbar(im5, ax=axes[2, 1])

plt.tight_layout()
plt.show()